In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import string
from itertools import product
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from transformers import BertForSequenceClassification, BertTokenizer
import torch
from torch.utils.data import TensorDataset, DataLoader, RandomSampler
from transformers import AdamW
import matplotlib.pyplot as plt

In [ ]:
# Carregar os dados originais
df_nomes = pd.read_csv('nomes_pessoais.csv')
df_nomes['y'] = 1

df_palavras = pd.read_csv('palavras_comuns.csv').sample(n=102561, random_state=42)
df_palavras['y'] = 0

df = pd.concat([df_nomes, df_palavras])
df.shape


In [ ]:
# Carregar o arquivo gabarito_formatado.csv
unique_words_df = pd.read_csv('gabarito_formatado.csv')

# Remover valores NaN na coluna 'palavra'
unique_words_df = unique_words_df.dropna(subset=['palavra'])

# Converter todos os valores '-1' para '0' na coluna 'y'
unique_words_df['y'] = unique_words_df['y'].replace(-1, 0)

# Separar palavras e nomes
words_df = unique_words_df[unique_words_df['y'] == 0]
names_df = unique_words_df[unique_words_df['y'] == 1]

# Samplear aleatoriamente 743 palavras
words_sample_df = words_df.sample(n=743, random_state=42)

# Concatenar as palavras sampleadas com todos os nomes
balanced_test_df = pd.concat([words_sample_df, names_df])

# Shuffle o dataframe para misturar as palavras e os nomes
balanced_test_df = balanced_test_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Verificar a distribuição de rótulos no conjunto de teste balanceado
label_distribution = balanced_test_df['y'].value_counts()
print("Distribuição de rótulos no conjunto de teste balanceado:")
print(label_distribution)

balanced_test_df

In [ ]:
# Carregar o modelo BERTimbau pré-treinado e o tokenizer
model_name = 'neuralmind/bert-base-portuguese-cased'
bert_model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)
tokenizer = BertTokenizer.from_pretrained(model_name)


In [ ]:
# Função para tokenizar os dados
def tokenize_data(data):
    return tokenizer(data, padding=True, truncation=True, return_tensors='pt')

In [ ]:
# Tokenizar os dados de treino
X_train_tokens = tokenize_data(df['X'].tolist())
y_train = torch.tensor(df['y'].values)

# Tokenizar as palavras únicas para teste
X_unique_words_tokens = tokenize_data(balanced_test_df['palavra'].tolist())
y_true = torch.tensor(balanced_test_df['y'].values)

In [ ]:
# Criar DataLoader para o conjunto de treino
train_dataset = TensorDataset(X_train_tokens['input_ids'], X_train_tokens['attention_mask'], y_train)
train_sampler = RandomSampler(train_dataset)
train_dataloader = DataLoader(train_dataset, sampler=train_sampler, batch_size=16)

# Treinar o modelo BERTimbau
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model.to(device)
optimizer = AdamW(bert_model.parameters(), lr=2e-5)

In [ ]:
for epoch in range(10):  # Ajuste o número de épocas conforme necessário
    bert_model.train()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    print(f'Época {epoch + 1}/10:')
    for batch_idx, batch in enumerate(train_dataloader):
        optimizer.zero_grad()
        input_ids, attention_mask, labels = batch
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)
        outputs = bert_model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)
        total_correct += torch.sum(preds == labels).item()
        total_samples += len(labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        # Log a cada 10 batches
        if (batch_idx + 1) % 10 == 0:
            avg_loss = total_loss / (batch_idx + 1)
            avg_accuracy = total_correct / total_samples
            print(f'    Batch {batch_idx + 1}/{len(train_dataloader)} - Loss: {avg_loss:.4f}, Acurácia: {avg_accuracy:.4f}')

In [ ]:
# Avaliar o modelo BERTimbau
bert_model.eval()
with torch.no_grad():
    input_ids = X_unique_words_tokens['input_ids'].to(device)
    attention_mask = X_unique_words_tokens['attention_mask'].to(device)
    outputs = bert_model(input_ids, attention_mask=attention_mask)
    logits = outputs.logits
    y_pred = torch.argmax(logits, dim=1).cpu().numpy()

In [ ]:
# Calcular as métricas
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")

In [ ]:
# Gerar a matriz de confusão
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot(cmap='binary')
plt.title('BERTimbau BASE')
plt.show()

In [ ]:
unique_words_df.info('y')